In [2]:
# ============================================================
# HEART DISEASE DATASET – FULL EDA & PREPROCESSING PIPELINE
# Works for Switzerland, Cleveland, Hungarian, Long-Beach-VA
# Handles messy rows, inconsistent columns, and -9 missing values
# ============================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. LOAD THE DATASET (robust loader)
# ------------------------------------------------------------

df = pd.read_csv(
    "switzerland.csv",
    delim_whitespace=True,   # handles irregular spacing
    header=None,             # no column names
    engine="python",
    on_bad_lines="skip"      # skip malformed rows instead of crashing
)

# Replace -9 with NaN
df = df.replace(-9, np.nan)

print("First 5 rows:")
display(df.head())

print("\nShape:", df.shape)

# ------------------------------------------------------------
# 2. BASIC EDA
# ------------------------------------------------------------

print("\nSummary statistics:")
display(df.describe())

print("\nMissing values per column:")
print(df.isna().sum())

# ------------------------------------------------------------
# 3. HANDLE MISSING DATA
# ------------------------------------------------------------

num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Fill numerical missing values with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Drop columns with too many missing values
df = df.loc[:, df.isna().mean() < 0.5]

print("\nMissing values after cleaning:")
print(df.isna().sum())

# ------------------------------------------------------------
# 4. OUTLIER HANDLING (95th percentile cap)
# ------------------------------------------------------------

for col in num_cols:
    cap = df[col].quantile(0.95)
    df[col] = np.where(df[col] > cap, cap, df[col])

# ------------------------------------------------------------
# 5. ENCODE CATEGORICAL VARIABLES
# ------------------------------------------------------------

df = pd.get_dummies(df, drop_first=True)

# ------------------------------------------------------------
# 6. FEATURE SCALING
# ------------------------------------------------------------

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ------------------------------------------------------------
# 7. TRAIN/TEST SPLIT (70/30)
# ------------------------------------------------------------

# Use last column as target (common in UCI heart datasets)
target_col = df.columns[-1]

X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nPipeline completed successfully.")

First 5 rows:


/tmp/ipykernel_12820/1212546886.py:18: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


,0,1,2,3,4,5,6
0,3001.0,0.0,65.0,1.0,1,1.0,1.0
1,1.0,1.0,75.0,NaN,name,NaN,NaN
2,3002.0,0.0,32.0,1.0,0,0.0,0.0
3,5.0,1.0,63.0,NaN,name,NaN,NaN
4,3003.0,0.0,61.0,1.0,1,1.0,1.0



Shape: (246, 7)

Summary statistics:


,0,1,2,3,5,6
count,229.000000,233.000000,229.000000,123.000000,123.000000,123.000000
mean,1948.231441,0.703863,59.301310,0.918699,0.821138,0.780488
std,1847.598219,0.896753,9.997853,0.274414,0.384804,0.415609
min,1.000000,0.000000,32.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,53.000000,1.000000,1.000000,1.000000
50%,3009.000000,0.000000,60.000000,1.000000,1.000000,1.000000
75%,4016.000000,1.000000,66.000000,1.000000,1.000000,1.000000
max,4074.000000,4.000000,86.000000,1.000000,1.000000,1.000000



Missing values per column:
0     17
1     13
2     17
3    123
4      0
5    123
6    123
dtype: int64

Missing values after cleaning:
0    0
1    0
2    0
3    0
4    0
5    0
6    0
dtype: int64

Shapes:
X_train: (172, 7)
X_test: (74, 7)
y_train: (172,)
y_test: (74,)

Pipeline completed successfully.
